# MobileOne — An Improved One Millisecond Mobile Backbone

**Paper:** Vasu, P. K. A., Gabriel, J., Zhu, J., Tuzel, O., & Ranjan, A. (2023). MobileOne: An Improved One millisecond Mobile Backbone. *CVPR 2023*. https://arxiv.org/abs/2206.04040

**Implementation:** `hale_core/nn/vision/cnn/mobile_one.py`

> MobileOne uses *structural re-parameterisation*: train with multiple parallel branches for expressiveness, then fuse all branches into a single conv at inference — achieving both high accuracy and sub-millisecond latency on mobile hardware.

## Architecture Overview

```
TRAINING MODE (MobileOneBlock, k branches):

Input x [B, C_in, H, W]
       │
  ┌────┼────────────────────────────────────────────────┐
  │    │  DEPTHWISE STAGE                               │
  │    ├─ 3×3 DW conv + BN  (×k branches)              │
  │    ├─ 1×1 DW conv + BN  (skip-like)                │
  │    └─ BN only           (identity, if same dims)    │
  │    → SUM all branch outputs → ReLU                  │
  └────────────────────────────────────────────────────┘
       │
  ┌────┼────────────────────────────────────────────────┐
  │    │  POINTWISE STAGE                              │
  │    ├─ 1×1 PW conv + BN  (×k branches)              │
  │    └─ BN only           (identity, if same dims)    │
  │    → SUM → ReLU                                     │
  └────────────────────────────────────────────────────┘

INFERENCE MODE (after reparametrize()):

Input x [B, C_in, H, W]
  → reparam_dw (single fused 3×3 DW conv + bias)
  → ReLU
  → reparam_pw (single fused 1×1 PW conv + bias)
  → ReLU
  → Output [B, C_out, H, W]

BN Fusion Formula:
  W_fused = (γ/√(σ²+ε)) · W_conv       scale per output channel
  b_fused = β - γ·μ/√(σ²+ε)           shift per output channel
  Sum over all k branches + pad 1×1→3×3 + add identity branch
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(17, 11))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.axis('off')

def box(ax, x, y, w, h, text, sub='', fc='#1f6feb', fontsize=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle='round,pad=0.07',
                        facecolor=fc, edgecolor='#58a6ff', lw=1.3)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2+(0.08 if sub else 0), text, ha='center', va='center',
            color='white', fontsize=fontsize, fontweight='bold', fontfamily='monospace')
    if sub:
        ax.text(x+w/2, y+h/2-0.17, sub, ha='center', va='center',
                color='#8b949e', fontsize=6.5)

def arr(ax, x1,y1,x2,y2,c='#58a6ff'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=c, lw=1.4))

# ── LEFT: Training mode multi-branch ──────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 12)
ax.set_ylim(0, 14)
ax.text(6, 13.5, 'Training Mode (k=4 branches)', ha='center',
        color='#f0f6fc', fontsize=11, fontweight='bold')

box(ax, 4, 12.5, 4, 0.6, 'Input x', '[B, C_in, H, W]', fc='#21262d')
ax.text(6, 12.1, '▼ Depthwise Stage', ha='center', color='#58a6ff', fontsize=9, fontstyle='italic')

# 4 branches depthwise
branch_colors = ['#1f6feb','#388bfd','#0d47a1','#1565c0']
for i in range(4):
    bx = 0.3 + i*2.8
    box(ax, bx, 10.5, 2.5, 0.8, f'DW 3×3\n+BN #{i+1}', '', fc=branch_colors[i], fontsize=8)
    arr(ax, 6, 12.5, bx+1.25, 11.3)

# 1x1 skip
box(ax, 9.5, 10.5, 2.3, 0.8, '1×1 DW\n+BN (skip)', '', fc='#6e40c9')
arr(ax, 6, 12.5, 10.6, 11.3)

# identity BN
box(ax, 3.5, 9.1, 2.5, 0.7, 'BN only', 'identity branch', fc='#2ea043')

# Σ
box(ax, 4.5, 8.0, 3, 0.7, 'SUM all branches', '+ identity', fc='#9e6a03')
for i in range(4):
    arr(ax, 0.3+i*2.8+1.25, 10.5, 6, 8.7)
arr(ax, 10.6, 10.5, 7.5, 8.7)
arr(ax, 4.75, 9.1, 6, 8.7)

box(ax, 4.5, 7.0, 3, 0.7, 'ReLU', '', fc='#388bfd')
arr(ax, 6, 8.0, 6, 7.7)

ax.text(6, 6.6, '▼ Pointwise Stage', ha='center', color='#58a6ff', fontsize=9, fontstyle='italic')

for i in range(4):
    bx = 0.3 + i*2.8
    box(ax, bx, 5.0, 2.5, 0.8, f'PW 1×1\n+BN #{i+1}', '', fc=branch_colors[i], fontsize=8)
    arr(ax, 6, 7.0, bx+1.25, 5.8)

box(ax, 4.5, 3.7, 3, 0.7, 'SUM branches', '', fc='#9e6a03')
for i in range(4):
    arr(ax, 0.3+i*2.8+1.25, 5.0, 6, 4.4)

box(ax, 4.5, 2.7, 3, 0.7, 'ReLU', '', fc='#388bfd')
arr(ax, 6, 3.7, 6, 3.4)
box(ax, 4.5, 1.7, 3, 0.7, 'Output', '[B, C_out, H, W]', fc='#21262d')
arr(ax, 6, 2.7, 6, 2.4)

# ── RIGHT: Inference mode + reparameterisation math ──────────────────────────
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 14)
ax.text(5, 13.5, 'Inference Mode (after reparametrize())', ha='center',
        color='#f0f6fc', fontsize=11, fontweight='bold')

box(ax, 2.5, 12.3, 5, 0.7, 'Input x', '[B, C_in, H, W]', fc='#21262d')
box(ax, 2.5, 11.0, 5, 0.9, 'reparam_dw', 'single 3×3 DW Conv + bias', fc='#2ea043')
arr(ax, 5, 12.3, 5, 11.9)
box(ax, 2.5, 9.8, 5, 0.8, 'ReLU', '', fc='#388bfd')
arr(ax, 5, 11.0, 5, 10.6)
box(ax, 2.5, 8.6, 5, 0.9, 'reparam_pw', 'single 1×1 PW Conv + bias', fc='#2ea043')
arr(ax, 5, 9.8, 5, 9.5)
box(ax, 2.5, 7.4, 5, 0.8, 'ReLU → Output', '[B, C_out, H, W]', fc='#21262d')
arr(ax, 5, 8.6, 5, 8.2)

# BN fusion math
ax.text(5, 6.9, 'BN Fusion Mathematics', ha='center', color='#f0f6fc',
        fontsize=10, fontweight='bold')

lines = [
    ('For each branch with (W, BN: γ,β,μ,σ²):', '#f0f6fc'),
    ('  scale = γ / √(σ² + ε)', '#58a6ff'),
    ('  W_fused = W · scale[:,-,-,-]', '#58a6ff'),
    ('  b_fused = β - μ · scale', '#58a6ff'),
    ('', '#8b949e'),
    ('Identity branch (DW):', '#f0f6fc'),
    ('  W = eye kernel: 1 at [c,0,1,1], else 0', '#388bfd'),
    ('  Then fuse its BN → (W_id, b_id)', '#388bfd'),
    ('', '#8b949e'),
    ('1×1 skip → pad to 3×3 (F.pad [1,1,1,1])', '#6e40c9'),
    ('', '#8b949e'),
    ('Final: W_total = Σ W_fused_i (all branches)', '#2ea043'),
    ('       b_total = Σ b_fused_i', '#2ea043'),
    ('Result: 1 conv with W_total, b_total', '#f85149'),
    ('⟹ Mathematically identical to k branches!', '#f0f6fc'),
]
for i, (line, color) in enumerate(lines):
    ax.text(1.0, 6.4 - i*0.42, line, color=color, fontsize=7.8, fontfamily='monospace')

plt.suptitle('MobileOne: Structural Re-Parameterisation', fontsize=14,
             color='#f0f6fc', fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


## Key Innovations

### 1. The Mobile Latency Problem
Mobile models are bottlenecked not just by FLOPs, but by **memory access costs** and **activation functions** (which require reading/writing entire feature maps). Prior mobile architectures (MobileNetV3, EfficientNet-Lite) suffered from:
- Many activation layers → high memory bandwidth
- Batch Norm at train time adds inference overhead if not fused
- Multi-branch designs are needed during training but expensive at inference

### 2. Structural Re-Parameterisation (RepVGG, ECCV 2022)
The key insight: **a sum of linear operations on the same input is another linear operation**. Since `BN(Conv(x)) = W'x + b'` (a linear function of x), and the sum of linear functions is linear, all branches can be fused into one.

MobileOne extends RepVGG to the depthwise-pointwise factorisation:
- **Depthwise stage**: k branches of 3×3 DW + BN, 1 branch of 1×1 DW + BN, 1 identity BN
- **Pointwise stage**: k branches of 1×1 PW + BN, 1 identity BN

### 3. BN Fusion Derivation
For a conv with weight W and BN (γ, β, μ, σ²):
```
BN(Conv(x)) = γ·(W⊛x − μ)/√(σ²+ε) + β
            = (γ/√(σ²+ε))·W ⊛ x + (β − γμ/√(σ²+ε))
            = W_fused ⊛ x + b_fused
```
For k branches: sum = `(Σ W_fused_i) ⊛ x + Σ b_fused_i`

For the 1×1 skip branch: pad the 1×1 weight to 3×3 by adding zeros around the centre — then fuse normally.

For the identity BN: the equivalent 3×3 DW conv weight is a delta kernel (1 at centre, 0 elsewhere).

### 4. k Branches During Training
More branches → more capacity → better accuracy at equal inference cost. k=4 is used in practice. After training, call `reparametrize()` to fold all branches. The model then runs as a simple sequence of DW+PW convolutions.

### 5. Hardware Characteristics
MobileOne was specifically designed for Apple Neural Engine (ANE) and CoreML. The single-branch inference path (no skip connections, no branching) maximises hardware parallelism on accelerators that are optimised for linear pipelines.

## Component Walkthrough

### `_fuse_bn` helper (lines 62–87 in `mobile_one.py`)
```python
def _fuse_bn(conv, bn):
    w    = conv.weight                        # [C_out, C_in/G, kH, kW]
    scale = bn.weight / (bn.running_var + bn.eps).sqrt()  # γ/σ per channel
    w_fused = w * scale.view(-1, 1, 1, 1)    # scale each output channel's kernel
    b_fused = bn.bias - bn.running_mean * scale  # β - μγ/σ
    return w_fused, b_fused
```

### `MobileOneBlock.reparametrize` (lines 205–273 in `mobile_one.py`)
```python
def reparametrize(self):
    # Accumulate all depthwise branches
    dw_w = torch.zeros_like(self.dw_branches[0][0].weight)  # [C, 1, 3, 3]
    dw_b = torch.zeros(self.in_channels)
    for branch in self.dw_branches:
        w, b = _fuse_bn(branch[0], branch[1])
        dw_w += w;  dw_b += b
    if self.dw_skip_1x1:
        w, b = _fuse_bn(self.dw_skip_1x1[0], self.dw_skip_1x1[1])
        dw_w += F.pad(w, [1,1,1,1])  # pad 1×1 → 3×3
        dw_b += b
    if self.dw_bn:
        eye_w = _identity_dw_kernel(self.in_channels)  # delta kernel
        scale = self.dw_bn.weight / (self.dw_bn.running_var + self.dw_bn.eps).sqrt()
        dw_w += eye_w * scale[:, None, None, None]
        dw_b += self.dw_bn.bias - self.dw_bn.running_mean * scale
    self.reparam_dw = nn.Conv2d(..., bias=True)
    self.reparam_dw.weight.data = dw_w
    self.reparam_dw.bias.data   = dw_b
    # ... same for pointwise ...
    del self.dw_branches, self.dw_skip_1x1, self.dw_bn  # free memory
    self._is_reparam = True
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

x = torch.randn(1, 3, 224, 224)

bb = build_vision_backbone('mobile_one_s2', img_size=224)
bb.eval()
with torch.no_grad():
    out_train = bb(x)
print(f'Before reparametrize: pooled={tuple(out_train.pooled_output.shape)}')
print(f'  params: {bb.num_parameters():,}')

# Reparametrize (fuse all branches)
def reparametrize_model(model):
    from hale_core.nn.vision.cnn.mobile_one import MobileOneBlock
    for m in model.modules():
        if isinstance(m, MobileOneBlock):
            m.reparametrize()
    return model

bb_rep = reparametrize_model(bb)
bb_rep.eval()
with torch.no_grad():
    out_inf = bb_rep(x)
print(f'After reparametrize:  pooled={tuple(out_inf.pooled_output.shape)}')
print(f'  params: {bb_rep.num_parameters():,}  (fewer — branches deleted)')

# Verify outputs are identical
max_diff = (out_train.pooled_output - out_inf.pooled_output).abs().max().item()
print(f'\nMax difference between train/inference output: {max_diff:.2e}')
print('✓ Mathematically equivalent!' if max_diff < 1e-4 else '✗ Check fusion math')


## Model Variants

| Variant       | Width multipliers          | k  | Latency (ANE) | Top-1 (ImageNet) |
|---------------|----------------------------|----|---------------|------------------|
| mobile_one_s0 | [0.75, 1.0, 1.0, 2.0]     | 4  | 0.79 ms       | 71.4%            |
| mobile_one_s1 | [1.5,  1.5, 2.0, 2.5]     | 4  | 0.89 ms       | 75.9%            |
| mobile_one_s2 | [1.5,  2.0, 2.5, 4.0]     | 4  | 1.18 ms       | 77.4%            |
| mobile_one_s3 | [2.0,  2.5, 3.0, 4.0]     | 4  | 1.56 ms       | 78.1%            |
| mobile_one_s4 | [3.0,  3.5, 3.5, 4.0]     | 4  | 2.45 ms       | 79.4%            |

## References

1. **[Primary]** Vasu, P. K. A., Gabriel, J., Zhu, J., Tuzel, O., & Ranjan, A. (2023). **MobileOne: An Improved One millisecond Mobile Backbone**. *CVPR 2023*. https://arxiv.org/abs/2206.04040

2. Ding, X., Zhang, X., Ma, N., Han, J., Ding, G., & Sun, J. (2022). **RepVGG: Making VGG-style ConvNets Great Again**. *CVPR 2021*. — The structural re-parameterisation technique MobileOne is built on.

3. Howard, A., et al. (2019). **Searching for MobileNetV3**. *ICCV 2019*. — Prior mobile backbone that MobileOne improves upon in latency.

4. Ding, X., et al. (2022). **RepMLPNet: Hierarchical Vision MLP with Re-parameterised Mixing**. *CVPR 2022*. — Extends re-param to MLP blocks.

5. Chen, Y., et al. (2021). **Mobile-Former: Bridging MobileNet and Transformer**. *CVPR 2022*. — Hybrid mobile architecture using attention for global reasoning.